# ISL Recognition Pipeline â€” Kaggle Training Notebook

**Model**: minGRU + CTC (+ optional TUP)  
**Dataset**: iSign v1.1 (pre-extracted MediaPipe pose `.npy` files)  
**GPU**: T4 / P100 (set via **Settings â†’ Accelerator â†’ GPU**)

## Steps
1. Clone your GitHub repo  
2. Install extra dependencies  
3. Upload / mount your dataset  
4. Configure paths  
5. Run training  


## Step 1 â€” Verify GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
    !nvidia-smi

## Step 2 â€” Clone the Repository

In [ ]:
import os

REPO_URL = 'https://github.com/vengalsettihrishi/ISL-CorrectionPipeline.git'  # <-- update if different
REPO_DIR = '/kaggle/working/ISL-CorrectionPipeline'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls -la

## Step 3 â€” Install Dependencies

Kaggle already has PyTorch, NumPy, and pandas pre-installed.  
Only extra packages need to be installed.

In [ ]:
# Only install what Kaggle doesn't already have
!pip install -q mediapipe pyarrow fastparquet tqdm

## Step 4 â€” Mount / Upload Your Dataset

### Option A â€” Kaggle Dataset (Recommended)
1. Go to **kaggle.com â†’ Datasets â†’ New Dataset**
2. Upload your `data_iSign/` folder (CSV + poses/ folder with `.npy` files)
3. In this notebook: **Add data** â†’ search for your dataset
4. Set `DATA_DIR` below to `/kaggle/input/<your-dataset-name>/`

### Option B â€” Upload directly (small datasets only)
Upload via the **Upload** button on the right sidebar.

### Option C â€” Download from URL / Drive
Use the cell below if your data is hosted somewhere.

In [ ]:
import os, shutil
from pathlib import Path

# ============================================================
#  CONFIGURE THESE PATHS FOR YOUR DATASET
# ============================================================

# If you added a Kaggle Dataset, change this to your dataset path:
#   e.g.  /kaggle/input/isign-v1-poses
KAGGLE_DATASET_PATH = '/kaggle/input/isl-processed-full-dataset'  # <-- CHANGE ME

# Where the code expects the data (matches config.py defaults)
DATA_DIR = '/kaggle/working/ISL-CorrectionPipeline/processed_full_dataset'
os.makedirs(DATA_DIR, exist_ok=True)

# --- Symlink or copy from Kaggle input to working directory ---
if os.path.exists(KAGGLE_DATASET_PATH):
    print('Dataset found at:', KAGGLE_DATASET_PATH)
    print('Contents:', os.listdir(KAGGLE_DATASET_PATH))

    # Create symlinks so the code can find the files
    for item in os.listdir(KAGGLE_DATASET_PATH):
        src = os.path.join(KAGGLE_DATASET_PATH, item)
        dst = os.path.join(DATA_DIR, item)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            print(f'Linked {src} -> {dst}')
        else:
            print(f'Already exists: {dst}')
else:
    print(f'WARNING: Dataset not found at {KAGGLE_DATASET_PATH}')
    print('Please add your dataset via the Add Data button or update KAGGLE_DATASET_PATH.')

# Check structure
print('\nprocessed_full_dataset contents:')
for p in Path(DATA_DIR).iterdir():
    print(' ', p)

## Step 5 â€” (Optional) Convert Parquet â†’ NPY

**Skip this step** if you already have pre-extracted `.npy` pose files in `data_iSign/poses/`.

In [ ]:
POSES_DIR = os.path.join(DATA_DIR, 'train')

if os.path.exists(POSES_DIR) and len(list(Path(POSES_DIR).glob('*.npy'))) > 0:
    n_files = len(list(Path(POSES_DIR).glob('*.npy')))
    print(f'Found {n_files} pre-extracted .npy files. Skipping conversion.')
else:
    print('No .npy files found. Running convert_parquet_to_npy.py ...')
    # Point to your parquet files
    PARQUET_DIR = os.path.join(DATA_DIR, 'parquets')  # adjust if needed
    !python convert_parquet_to_npy.py \
        --parquet_dir {PARQUET_DIR} \
        --output_dir {POSES_DIR}

## Step 6 â€” Configure Training Hyperparameters

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from config import Config

cfg = Config()

# -----------------------------------------------------------
#  Paths (update if your data is elsewhere)
# -----------------------------------------------------------
cfg.data_dir   = DATA_DIR
cfg.csv_path   = os.path.join(DATA_DIR, 'manifest.csv')
cfg.pose_dir   = os.path.join(DATA_DIR, 'train')
cfg.video_dir  = os.path.join(DATA_DIR, 'videos')
cfg.vocab_path = os.path.join(DATA_DIR, 'vocab.json')
cfg.norm_stats_path = os.path.join(DATA_DIR, 'norm_stats.npz')
cfg.checkpoint_dir  = '/kaggle/working/checkpoints'

# -----------------------------------------------------------
#  Hyperparameters  (tune as needed)
# -----------------------------------------------------------
cfg.epochs       = 100
cfg.batch_size   = 32          # reduce to 16 if OOM on T4
cfg.learning_rate = 3e-4
cfg.hidden_size  = 128
cfg.num_gru_layers = 3
cfg.dropout      = 0.1
cfg.patience     = 15
cfg.seed         = 42

# TUP (Temporal Uncertainty Propagation) â€” set True to enable
cfg.enable_tup   = False
cfg.enable_velocity_temperature = False

# DataLoader workers
cfg.num_workers  = 2
cfg.pin_memory   = True
cfg.use_mixed_precision = True

print('Config validated:')
print(f'  device         = {cfg.device}')
print(f'  feature_dim    = {cfg.feature_dim}')
print(f'  batch_size     = {cfg.batch_size}')
print(f'  epochs         = {cfg.epochs}')
print(f'  checkpoint_dir = {cfg.checkpoint_dir}')
print(f'  pose_dir       = {cfg.pose_dir}')
os.makedirs(cfg.checkpoint_dir, exist_ok=True)

## Step 7 â€” Verify Data Loading

In [ ]:
from data.dataset import create_dataloaders
from data.label_encoder import LabelEncoder

print('Creating dataloaders...')
train_loader, val_loader, test_loader, encoder = create_dataloaders(cfg)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')
print(f'Vocab size    : {encoder.vocab_size}')

# Peek at one batch
batch = next(iter(train_loader))
if batch is not None:
    features, labels, feat_lengths, label_lengths = batch
    print(f'\nSample batch:')
    print(f'  features shape     : {features.shape}')   # (B, T, 450)
    print(f'  labels shape       : {labels.shape}')     # (B, L)
    print(f'  feat_lengths shape : {feat_lengths.shape}')
    print(f'  label_lengths shape: {label_lengths.shape}')
else:
    print('WARNING: First batch is None â€” check your data!')

## Step 8 â€” Run Training

In [ ]:
from train import train

results = train(cfg)

print('\n=== Training Complete ===')
print(f'Best Val WER  : {results["best_val_wer"]*100:.2f}%')
print(f'Best Val Loss : {results["best_val_loss"]:.4f}')
print(f'Vocab size    : {results["vocab_size"]}')

## Step 9 â€” Plot Training History

In [ ]:
import json, matplotlib.pyplot as plt

history_path = os.path.join(cfg.checkpoint_dir, 'history.json')
with open(history_path) as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total loss
axes[0].plot(epochs, history['train_loss'], label='Train Loss')
axes[0].plot(epochs, history['val_loss'],   label='Val Loss')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# CTC loss
axes[1].plot(epochs, history['train_ctc_loss'], label='Train CTC')
axes[1].plot(epochs, history['val_ctc_loss'],   label='Val CTC')
axes[1].set_title('CTC Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

# WER
axes[2].plot(epochs, [w * 100 for w in history['val_wer']], color='crimson', label='Val WER %')
axes[2].set_title('Validation WER')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('WER (%)')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()
print('Saved: /kaggle/working/training_curves.png')

## Step 10 â€” Save Outputs

All outputs under `/kaggle/working/` are automatically saved and downloadable.

In [ ]:
print('Output files:')
for p in Path('/kaggle/working').rglob('*'):
    if p.is_file():
        size = p.stat().st_size
        print(f'  {p.relative_to("/kaggle/working")}  ({size/1e6:.2f} MB)')

## (Optional) Retrain with TUP Enabled

Temporal Uncertainty Propagation lets the model learn which frames are unreliable.

In [ ]:
from config import Config
from train import train

cfg_tup = Config()
cfg_tup.data_dir   = DATA_DIR
cfg_tup.csv_path   = os.path.join(DATA_DIR, 'manifest.csv')
cfg_tup.pose_dir   = os.path.join(DATA_DIR, 'train')
cfg_tup.vocab_path = os.path.join(DATA_DIR, 'vocab.json')
cfg_tup.norm_stats_path = os.path.join(DATA_DIR, 'norm_stats.npz')
cfg_tup.checkpoint_dir  = '/kaggle/working/checkpoints_tup'

# Enable TUP
cfg_tup.enable_tup  = True
cfg_tup.tup_lambda  = 0.01
cfg_tup.tup_smooth_lambda = 0.01
cfg_tup.tup_target_activity = 0.6
cfg_tup.tup_hard_mask = True
cfg_tup.tup_blank_bias = 4.0

cfg_tup.epochs     = 100
cfg_tup.batch_size = 32
cfg_tup.num_workers = 2

os.makedirs(cfg_tup.checkpoint_dir, exist_ok=True)

results_tup = train(cfg_tup)
print(f'Best Val WER (TUP): {results_tup["best_val_wer"]*100:.2f}%')